In [9]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import ipaddress
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)

DATA_DIR = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\raw\cicids_v3")
DUCKDB_PATH = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\interim\cicids_eda.duckdb")
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

csv_files = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files:\n")
for f in csv_files:
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}  ({size_mb:.1f} MB)")

Found 8 CSV files:

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  (96.1 MB)
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  (101.9 MB)
  Friday-WorkingHours-Morning.pcap_ISCX.csv  (75.4 MB)
  Monday-WorkingHours.pcap_ISCX.csv  (268.6 MB)
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  (108.7 MB)
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  (92.0 MB)
  Tuesday-WorkingHours.pcap_ISCX.csv  (174.7 MB)
  Wednesday-workingHours.pcap_ISCX.csv  (285.6 MB)


In [ ]:
# Find all CSVs
csv_files = sorted(DATA_DIR.rglob("*.csv"))
con = duckdb.connect(str(DUCKDB_PATH))

print(f"{'File':<70} {'Rows':>10}")
print("-" * 85)
for f in csv_files:
    cnt = con.execute(f"""
        SELECT COUNT(*) FROM read_csv_auto(
            '{f.as_posix()}',
            sample_size=100000,
            ignore_errors=true,
            null_padding=true,
            all_varchar=true
        )
    """).fetchone()[0]
    print(f"{f.name:<70} {cnt:>10,}")

File                                                                         Rows
-------------------------------------------------------------------------------------
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                          225,745
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv                      286,467
Friday-WorkingHours-Morning.pcap_ISCX.csv                                 191,033
Monday-WorkingHours.pcap_ISCX.csv                                         529,918
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv               288,602
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv                    458,968
Tuesday-WorkingHours.pcap_ISCX.csv                                        445,909
Wednesday-workingHours.pcap_ISCX.csv                                      692,703


In [10]:
# Use Monday (smallest attack-free file) for schema inspection
monday = next(f for f in csv_files if 'Monday' in f.name)
print(f"Inspecting: {monday.name}\n")

# Just the header
header_df = pd.read_csv(monday, nrows=0)
print(f"Total columns: {len(header_df.columns)}")
print(f"\nColumn names (note leading spaces — quirk of dataset):")
for i, col in enumerate(header_df.columns):
    print(f"  {i:2d}: {repr(col)}")  # repr shows leading spaces

Inspecting: Monday-WorkingHours.pcap_ISCX.csv

Total columns: 85

Column names (note leading spaces — quirk of dataset):
   0: 'Flow ID'
   1: ' Source IP'
   2: ' Source Port'
   3: ' Destination IP'
   4: ' Destination Port'
   5: ' Protocol'
   6: ' Timestamp'
   7: ' Flow Duration'
   8: ' Total Fwd Packets'
   9: ' Total Backward Packets'
  10: 'Total Length of Fwd Packets'
  11: ' Total Length of Bwd Packets'
  12: ' Fwd Packet Length Max'
  13: ' Fwd Packet Length Min'
  14: ' Fwd Packet Length Mean'
  15: ' Fwd Packet Length Std'
  16: 'Bwd Packet Length Max'
  17: ' Bwd Packet Length Min'
  18: ' Bwd Packet Length Mean'
  19: ' Bwd Packet Length Std'
  20: 'Flow Bytes/s'
  21: ' Flow Packets/s'
  22: ' Flow IAT Mean'
  23: ' Flow IAT Std'
  24: ' Flow IAT Max'
  25: ' Flow IAT Min'
  26: 'Fwd IAT Total'
  27: ' Fwd IAT Mean'
  28: ' Fwd IAT Std'
  29: ' Fwd IAT Max'
  30: ' Fwd IAT Min'
  31: 'Bwd IAT Total'
  32: ' Bwd IAT Mean'
  33: ' Bwd IAT Std'
  34: ' Bwd IAT Max'
  35:

In [11]:
def clean_columns(df):
    """Strip whitespace and normalize column names to snake_case."""
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('/', '_')
        .str.replace('.', '_', regex=False)
        .str.replace('-', '_')
    )
    return df

# Quick peek at cleaned columns
sample = pd.read_csv(monday, nrows=100)
sample = clean_columns(sample)
print(f"Cleaned columns ({len(sample.columns)}):")
for c in sample.columns:
    print(f"  {c}")

Cleaned columns (85):
  flow_id
  source_ip
  source_port
  destination_ip
  destination_port
  protocol
  timestamp
  flow_duration
  total_fwd_packets
  total_backward_packets
  total_length_of_fwd_packets
  total_length_of_bwd_packets
  fwd_packet_length_max
  fwd_packet_length_min
  fwd_packet_length_mean
  fwd_packet_length_std
  bwd_packet_length_max
  bwd_packet_length_min
  bwd_packet_length_mean
  bwd_packet_length_std
  flow_bytes_s
  flow_packets_s
  flow_iat_mean
  flow_iat_std
  flow_iat_max
  flow_iat_min
  fwd_iat_total
  fwd_iat_mean
  fwd_iat_std
  fwd_iat_max
  fwd_iat_min
  bwd_iat_total
  bwd_iat_mean
  bwd_iat_std
  bwd_iat_max
  bwd_iat_min
  fwd_psh_flags
  bwd_psh_flags
  fwd_urg_flags
  bwd_urg_flags
  fwd_header_length
  bwd_header_length
  fwd_packets_s
  bwd_packets_s
  min_packet_length
  max_packet_length
  packet_length_mean
  packet_length_std
  packet_length_variance
  fin_flag_count
  syn_flag_count
  rst_flag_count
  psh_flag_count
  ack_flag_count
  

In [12]:
# Categorize columns by usefulness
identification_cols = [
    'flow_id',                 # unique flow identifier (if present)
    'source_ip',               # for geo-mapping (if present in your version)
    'source_port',
    'destination_ip',          # for geo-mapping (if present)
    'destination_port',
    'protocol',
    'timestamp',               # for time-series
]

volumetric_cols = [
    'flow_duration',
    'total_fwd_packets',
    'total_backward_packets',
    'total_length_of_fwd_packets',
    'total_length_of_bwd_packets',
    'flow_bytes_s',
    'flow_packets_s',
]

behavioral_cols = [
    'fwd_packet_length_mean',
    'bwd_packet_length_mean',
    'flow_iat_mean',           # inter-arrival time
    'fwd_iat_mean',
    'bwd_iat_mean',
    'syn_flag_count',
    'rst_flag_count',
    'ack_flag_count',
]

label_col = ['label']

# Check which exist in your file
all_target = identification_cols + volumetric_cols + behavioral_cols + label_col
present = [c for c in all_target if c in sample.columns]
missing = [c for c in all_target if c not in sample.columns]

print(f"Present ({len(present)}):")
for c in present: print(f"  ✅ {c}")
print(f"\nMissing ({len(missing)}):")
for c in missing: print(f"  ❌ {c}")

Present (23):
  ✅ flow_id
  ✅ source_ip
  ✅ source_port
  ✅ destination_ip
  ✅ destination_port
  ✅ protocol
  ✅ timestamp
  ✅ flow_duration
  ✅ total_fwd_packets
  ✅ total_backward_packets
  ✅ total_length_of_fwd_packets
  ✅ total_length_of_bwd_packets
  ✅ flow_bytes_s
  ✅ flow_packets_s
  ✅ fwd_packet_length_mean
  ✅ bwd_packet_length_mean
  ✅ flow_iat_mean
  ✅ fwd_iat_mean
  ✅ bwd_iat_mean
  ✅ syn_flag_count
  ✅ rst_flag_count
  ✅ ack_flag_count
  ✅ label

Missing (0):


In [20]:
con = duckdb.connect(str(DUCKDB_PATH))

# Drop the old tables to start clean
existing_tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]
for t in existing_tables:
    print(f"Dropping {t}...")
    con.execute(f"DROP TABLE IF EXISTS {t}")

print("\n✅ All old tables dropped\n")

Dropping raw_friday...
Dropping raw_monday...
Dropping raw_thursday...
Dropping raw_tuesday...
Dropping raw_wednesday...
Dropping stg_cicids...

✅ All old tables dropped



In [21]:
# Find all CSVs
csv_files = sorted(DATA_DIR.rglob("*.csv"))

# Map each file to a clean DuckDB table name
def get_table_name(filename):
    """Convert CSV filename to a clean DuckDB table name."""
    name = filename.replace('.pcap_ISCX.csv', '')
    name = name.replace('-WorkingHours', '')
    name = name.replace('-', '_').lower()
    return f"raw_{name}"

print(f"{'CSV File':<70} → {'Table Name'}")
print("-" * 100)
for f in csv_files:
    table_name = get_table_name(f.name)
    print(f"{f.name:<70} → {table_name}")

CSV File                                                               → Table Name
----------------------------------------------------------------------------------------------------
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                       → raw_friday_afternoon_ddos
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv                   → raw_friday_afternoon_portscan
Friday-WorkingHours-Morning.pcap_ISCX.csv                              → raw_friday_morning
Monday-WorkingHours.pcap_ISCX.csv                                      → raw_monday
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv            → raw_thursday_afternoon_infilteration
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv                 → raw_thursday_morning_webattacks
Tuesday-WorkingHours.pcap_ISCX.csv                                     → raw_tuesday
Wednesday-workingHours.pcap_ISCX.csv                                   → raw_wednesday_workinghours


In [24]:
def derive_source_day(filename):
    """Extract day-of-week from filename."""
    for day in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']:
        if day in filename:
            return day
    return 'Unknown'

# Load each file with explicit settings
load_results = []
for f in csv_files:
    table_name = get_table_name(f.name)
    source_day = derive_source_day(f.name)
    
    print(f"Loading {f.name}...")
    
    try:
        con.execute(f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT *, '{source_day}' AS source_day
            FROM read_csv(
                '{f.as_posix()}',
                header=true,
                ignore_errors=true,
                null_padding=true,
                all_varchar=true,
                sample_size=-1
            )
        """)
        cnt = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        cols = con.execute(f"SELECT COUNT(*) FROM (DESCRIBE {table_name})").fetchone()[0]
        load_results.append({
            'file': f.name, 'table': table_name, 'rows': cnt, 'cols': cols
        })
        print(f"  ✅ {table_name}: {cnt:,} rows, {cols} cols")
    except Exception as e:
        print(f"  ❌ Failed: {e}")
        load_results.append({
            'file': f.name, 'table': table_name, 'rows': 0, 'cols': 0, 'error': str(e)
        })

# Summary
import pandas as pd
print("\n=== Load Summary ===")
print(pd.DataFrame(load_results)[['file', 'table', 'rows', 'cols']])
print(f"\nTotal rows loaded: {sum(r['rows'] for r in load_results):,}")

Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅ raw_friday_afternoon_ddos: 225,745 rows, 86 cols
Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅ raw_friday_afternoon_portscan: 286,467 rows, 86 cols
Loading Friday-WorkingHours-Morning.pcap_ISCX.csv...


...

  ✅ raw_thursday_morning_webattacks: 456,788 rows, 86 cols
Loading Tuesday-WorkingHours.pcap_ISCX.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅ raw_tuesday: 445,909 rows, 86 cols
Loading Wednesday-workingHours.pcap_ISCX.csv...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅ raw_wednesday_workinghours: 692,703 rows, 86 cols

=== Load Summary ===
                                                          file                                 table    rows  cols
0             Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv             raw_friday_afternoon_ddos  225745    86
1         Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv         raw_friday_afternoon_portscan  286467    86
2                    Friday-WorkingHours-Morning.pcap_ISCX.csv                    raw_friday_morning  191033    86
3                            Monday-WorkingHours.pcap_ISCX.csv                            raw_monday  529918    86
4  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  raw_thursday_afternoon_infilteration  288602    86
5       Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv       raw_thursday_morning_webattacks  456788    86
6                           Tuesday-WorkingHours.pcap_ISCX.csv                           raw_tuesday  445909    86
7   

In [26]:
all_tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]

print("=== Label distribution per table ===\n")
for t in all_tables:
    print(f"📁 {t}")
    
    # Find the label column (case-insensitive search since loaded as varchar)
    cols = [r[0] for r in con.execute(f"DESCRIBE {t}").fetchall()]
    label_col = next((c for c in cols if c.strip().lower() == 'label'), None)
    
    if label_col is None:
        print(f"   ⚠️ No Label column found! Columns: {cols[:5]}...")
        continue
    
    dist = con.execute(f"""
        SELECT "{label_col}" AS label, COUNT(*) AS cnt
        FROM {t}
        GROUP BY "{label_col}"
        ORDER BY cnt DESC
    """).fetchdf()
    
    for _, row in dist.iterrows():
        label_display = repr(row['label'])  # repr shows whitespace and special chars
        print(f"   {label_display:<50} {row['cnt']:>10,}")
    print()

=== Label distribution per table ===

📁 raw_friday_afternoon_ddos
   'DDoS'                                                128,027
   'BENIGN'                                               97,718

📁 raw_friday_afternoon_portscan
   'PortScan'                                            158,930
   'BENIGN'                                              127,537

📁 raw_friday_morning
   'BENIGN'                                              189,067
   'Bot'                                                   1,966

📁 raw_monday
   'BENIGN'                                              529,918

📁 raw_thursday_afternoon_infilteration
   'BENIGN'                                              288,566
   'Infiltration'                                             36

📁 raw_thursday_morning_webattacks
   None                                                  288,602
   'BENIGN'                                              168,186

📁 raw_tuesday
   'BENIGN'                                              432

In [30]:
DATA_DIR = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\raw\cicids_v3")
problem_file = next(DATA_DIR.rglob("Thursday-WorkingHours-Morning-WebAttacks*"))

print(f"File: {problem_file.name}")
print(f"Size: {problem_file.stat().st_size / 1e6:.2f} MB\n")

# Read first 500 bytes raw to check for encoding weirdness
with open(problem_file, 'rb') as f:
    raw_bytes = f.read(500)
print("First 500 bytes (raw):")
print(raw_bytes)
print()

# Check for non-printable characters
non_printable = sum(1 for b in raw_bytes if b < 32 and b not in (9, 10, 13))
print(f"Non-printable bytes in first 500: {non_printable}")

# Count actual line endings in the whole file
with open(problem_file, 'rb') as f:
    content = f.read()
print(f"\nLine ending counts:")
print(f"  \\r\\n (Windows): {content.count(b'\\r\\n'):,}")
print(f"  \\n only (Unix): {content.count(b'\\n') - content.count(b'\\r\\n'):,}")
print(f"  \\r only (old Mac): {content.count(b'\\r') - content.count(b'\\r\\n'):,}")
print(f"  Null bytes: {content.count(b'\\x00'):,}")

File: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Size: 92.03 MB

First 500 bytes (raw):
b'Flow ID, Source IP, Source Port, Destination IP, Destination Port, Protocol, Timestamp, Flow Duration, Total Fwd Packets, Total Backward Packets,Total Length of Fwd Packets, Total Length of Bwd Packets, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std,Bwd Packet Length Max, Bwd Packet Length Min, Bwd Packet Length Mean, Bwd Packet Length Std,Flow Bytes/s, Flow Packets/s, Flow IAT Mean, Flow IAT Std, Flow IAT Max, Flow IAT Min,Fwd IAT Total, Fwd IAT Mean'

Non-printable bytes in first 500: 0

Line ending counts:
  \r\n (Windows): 0
  \n only (Unix): 0
  \r only (old Mac): 0
  Null bytes: 0


In [33]:
con = duckdb.connect(DUCKDB_PATH)

# Find the actual Label column name
cols = [r[0] for r in con.execute("DESCRIBE raw_thursday_morning_webattacks").fetchall()]
label_col = next(c for c in cols if c.strip().lower() == 'label')

print(f"Label column name: {repr(label_col)}\n")

# Show some NULL-label rows
print("=== Sample rows with NULL label ===")
sample = con.execute(f"""
    SELECT "Source IP", "Destination IP", "Source Port", "Destination Port", 
           "Protocol", "Timestamp", "{label_col}" AS label
    FROM raw_thursday_morning_webattacks
    WHERE "{label_col}" IS NULL
    LIMIT 5
""").fetchdf()
print(sample)

print("\n=== Sample rows with BENIGN label (for comparison) ===")
sample = con.execute(f"""
    SELECT "Source IP", "Destination IP", "Source Port", "Destination Port", 
           "Protocol", "Timestamp", "{label_col}" AS label
    FROM raw_thursday_morning_webattacks
    WHERE "{label_col}" = 'BENIGN'
    LIMIT 5
""").fetchdf()
print(sample)

Label column name: 'Label'

=== Sample rows with NULL label ===
  Source IP Destination IP Source Port Destination Port Protocol Timestamp label
0      None           None        None             None     None      None  None
1      None           None        None             None     None      None  None
2      None           None        None             None     None      None  None
3      None           None        None             None     None      None  None
4      None           None        None             None     None      None  None

=== Sample rows with BENIGN label (for comparison) ===
       Source IP Destination IP Source Port Destination Port Protocol      Timestamp   label
0  192.168.10.50   192.168.10.3       33898              389        6  6/7/2017 8:59  BENIGN
1  192.168.10.50   192.168.10.3       33904              389        6  6/7/2017 8:59  BENIGN
2        8.6.0.1        8.0.6.4           0                0        0  6/7/2017 8:59  BENIGN
3  192.168.10.14   65.

In [34]:
# Show all columns of the problem table — Label should be last
print("Columns in raw_thursday_morning_webattacks:")
all_cols = con.execute("DESCRIBE raw_thursday_morning_webattacks").fetchdf()
print(all_cols[['column_name', 'column_type']].tail(10))
print(f"\nTotal columns: {len(all_cols)}")

Columns in raw_thursday_morning_webattacks:
    column_name column_type
76  Active Mean     VARCHAR
77   Active Std     VARCHAR
78   Active Max     VARCHAR
79   Active Min     VARCHAR
80    Idle Mean     VARCHAR
81     Idle Std     VARCHAR
82     Idle Max     VARCHAR
83     Idle Min     VARCHAR
84        Label     VARCHAR
85   source_day     VARCHAR

Total columns: 86


In [35]:
from pathlib import Path

problem_file = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\raw\cicids_v3")
problem_file = next(problem_file.rglob("Thursday-WorkingHours-Morning-WebAttacks*"))

# Read the entire file as bytes and analyze
with open(problem_file, 'rb') as f:
    content = f.read()

print(f"File size: {len(content):,} bytes\n")

# Better line ending analysis
crlf = content.count(b'\r\n')
lf_only = content.count(b'\n') - crlf  # \n not part of \r\n
cr_only = content.count(b'\r') - crlf  # \r not part of \r\n

print(f"\\r\\n (Windows): {crlf:,}")
print(f"\\n only (Unix):   {lf_only:,}")
print(f"\\r only (old Mac): {cr_only:,}")

# Look for unusual characters
print(f"\nNon-ASCII bytes: {sum(1 for b in content if b > 127):,}")
print(f"Null bytes (\\x00): {content.count(0):,}")

# Find a sample line break and look around it
first_lf = content.find(b'\n')
print(f"\nFirst newline at byte {first_lf}")
if first_lf > 0:
    print(f"Bytes around first newline:")
    print(f"  Before: {content[max(0, first_lf-30):first_lf]!r}")
    print(f"  After:  {content[first_lf:first_lf+30]!r}")

# Check what character code precedes most newlines
import collections
preceding = collections.Counter()
i = content.find(b'\n')
while i != -1:
    if i > 0:
        preceding[content[i-1]] += 1
    i = content.find(b'\n', i+1)
print(f"\nTop 5 characters preceding newlines (byte_value: count):")
for byte_val, cnt in preceding.most_common(5):
    print(f"  {byte_val} ({chr(byte_val)!r}): {cnt:,}")

File size: 92,030,223 bytes

\r\n (Windows): 458,969
\n only (Unix):   0
\r only (old Mac): 0

Non-ASCII bytes: 2,180
Null bytes (\x00): 0

First newline at byte 1442
Bytes around first newline:
  Before: b'td, Idle Max, Idle Min, Label\r'
  After:  b'\n192.168.10.3-192.168.10.50-38'

Top 5 characters preceding newlines (byte_value: count):
  13 ('\r'): 458,969


In [36]:
problem_file_path = r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\raw\cicids_v3\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
# Adjust path to where your file actually lives — check Cell 4.5 output for exact path

# Try pandas with multiple encoding attempts
for encoding in ['utf-8', 'latin-1', 'cp1252']:
    try:
        df = pd.read_csv(
            problem_file_path,
            encoding=encoding,
            on_bad_lines='warn',  # warns about problem rows but continues
            low_memory=False,
        )
        print(f"✅ Successfully loaded with encoding={encoding}")
        print(f"   Shape: {df.shape}")
        break
    except Exception as e:
        print(f"❌ encoding={encoding} failed: {str(e)[:200]}")

# Strip whitespace from column names (CICIDS2017 has leading spaces)
df.columns = df.columns.str.strip()

# Check the Label column distribution
print("\n=== Label distribution from pandas load ===")
print(df['Label'].value_counts(dropna=False))

❌ encoding=utf-8 failed: 'utf-8' codec can't decode byte 0x96 in position 11: invalid start byte
✅ Successfully loaded with encoding=latin-1
   Shape: (458968, 85)

=== Label distribution from pandas load ===
Label
NaN                           288602
BENIGN                        168186
Web Attack  Brute Force        1507
Web Attack  XSS                 652
Web Attack  Sql Injection        21
Name: count, dtype: int64


In [37]:
# Step 1: Load with pandas
print(f"Loading {problem_file.name} with pandas (latin-1)...")
df = pd.read_csv(
    problem_file,
    encoding='latin-1',
    on_bad_lines='warn',
    low_memory=False,
)

# Step 2: Strip whitespace from column names (handles leading-space quirk)
df.columns = df.columns.str.strip()

# Step 3: Add source_day column
df['source_day'] = 'Thursday'

# Step 4: Convert all columns to string (matches `all_varchar=true` in other tables)
df = df.astype(str)

# Step 5: Replace 'nan' string (from astype(str) on NULLs) with actual NULL
df = df.replace('nan', None)

print(f"Loaded shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

# Step 6: Push to DuckDB, replacing the broken table
con = duckdb.connect(str(DUCKDB_PATH))

# Drop existing table
con.execute("DROP TABLE IF EXISTS raw_thursday_morning_webattacks")

# Register the dataframe and create new table
con.register('df_temp', df)
con.execute("""
    CREATE TABLE raw_thursday_morning_webattacks AS 
    SELECT * FROM df_temp
""")
con.unregister('df_temp')

# Verify
new_cnt = con.execute("SELECT COUNT(*) FROM raw_thursday_morning_webattacks").fetchone()[0]
new_cols = con.execute("SELECT COUNT(*) FROM (DESCRIBE raw_thursday_morning_webattacks)").fetchone()[0]
print(f"\n✅ Replaced table: {new_cnt:,} rows × {new_cols} cols")

# Confirm label distribution now matches pandas
print("\n=== New label distribution ===")
print(con.execute("""
    SELECT "Label", COUNT(*) AS cnt
    FROM raw_thursday_morning_webattacks
    GROUP BY "Label"
    ORDER BY cnt DESC
""").fetchdf())

Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv with pandas (latin-1)...
Loaded shape: (458968, 86)
Columns: 86


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✅ Replaced table: 458,968 rows × 86 cols

=== New label distribution ===
                        Label     cnt
0                        None  288602
1                      BENIGN  168186
2    Web Attack  Brute Force    1507
3            Web Attack  XSS     652
4  Web Attack  Sql Injection      21


# Unified Table

In [39]:
con = duckdb.connect(str(DUCKDB_PATH))

# Get all 8 raw tables in a stable order
all_tables = sorted([r[0] for r in con.execute("SHOW TABLES").fetchall()])
print(f"Tables to union: {len(all_tables)}")
for t in all_tables:
    cnt = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {cnt:,}")

# The 84 BI-relevant columns we want to keep (excluding Fwd Header Length_1 duplicate)
keep_cols = [
    'Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port',
    'Protocol', 'Timestamp',
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std',
    'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std',
    'Flow Bytes/s', 'Flow Packets/s',
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
    'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags',
    'Fwd Header Length', 'Bwd Header Length',
    'Fwd Packets/s', 'Bwd Packets/s',
    'Min Packet Length', 'Max Packet Length',
    'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
    'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count',
    'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count',
    'Down/Up Ratio', 'Average Packet Size',
    'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',
    'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate',
    'Subflow Fwd Packets', 'Subflow Fwd Bytes',
    'Subflow Bwd Packets', 'Subflow Bwd Bytes',
    'Init_Win_bytes_forward', 'Init_Win_bytes_backward',
    'act_data_pkt_fwd', 'min_seg_size_forward',
    'Active Mean', 'Active Std', 'Active Max', 'Active Min',
    'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min',
    'Label', 'source_day',
]
print(f"\nColumns to keep: {len(keep_cols)}")

Tables to union: 8
  raw_friday_afternoon_ddos: 225,745
  raw_friday_afternoon_portscan: 286,467
  raw_friday_morning: 191,033
  raw_monday: 529,918
  raw_thursday_afternoon_infilteration: 288,602
  raw_thursday_morning_webattacks: 458,968
  raw_tuesday: 445,909
  raw_wednesday_workinghours: 692,703

Columns to keep: 85


In [40]:
# Build the UNION query
quoted_cols = ', '.join([f'"{c}"' for c in keep_cols])

union_parts = [f'SELECT {quoted_cols} FROM {t}' for t in all_tables]
union_sql = ' UNION ALL '.join(union_parts)

# Build the staging table
con.execute(f"""
    CREATE OR REPLACE TABLE stg_cicids_raw AS
    {union_sql}
""")

total = con.execute("SELECT COUNT(*) FROM stg_cicids_raw").fetchone()[0]
print(f"✅ stg_cicids_raw materialized: {total:,} rows × {len(keep_cols)} cols")

# Compare to sum of individual tables
expected = sum(con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in all_tables)
print(f"   Expected: {expected:,} rows")
print(f"   Match: {'✅' if total == expected else '❌ MISMATCH'}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ stg_cicids_raw materialized: 3,119,345 rows × 85 cols
   Expected: 3,119,345 rows
   Match: ✅


In [41]:
# Apply label normalization, attack family categorization, and ML-ready binary flag
con.execute("""
    CREATE OR REPLACE TABLE stg_cicids AS
    SELECT *,
        -- Clean label: handle NULLs and the \\x96 character
        CASE
            WHEN "Label" IS NULL OR TRIM("Label") = '' THEN 'Unlabeled'
            ELSE TRIM(REGEXP_REPLACE("Label", '[^[:print:]]+', '-', 'g'))
        END AS label_clean,
        
        -- High-level attack family for dashboarding
        CASE
            WHEN "Label" IS NULL OR TRIM("Label") = '' THEN 'Unlabeled'
            WHEN "Label" ILIKE '%BENIGN%' THEN 'Benign'
            WHEN "Label" ILIKE '%DDoS%' THEN 'DDoS'
            WHEN "Label" ILIKE '%DoS%' THEN 'DoS'
            WHEN "Label" ILIKE '%Patator%' THEN 'Brute Force'
            WHEN "Label" ILIKE '%PortScan%' THEN 'Reconnaissance'
            WHEN "Label" ILIKE '%Web Attack%' THEN 'Web Attack'
            WHEN "Label" ILIKE '%Bot%' THEN 'Botnet'
            WHEN "Label" ILIKE '%Infiltration%' THEN 'Infiltration'
            WHEN "Label" ILIKE '%Heartbleed%' THEN 'Exploit'
            ELSE 'Other'
        END AS attack_family,
        
        -- Binary flag for ML (excludes Unlabeled from supervised training)
        CASE
            WHEN "Label" IS NULL OR TRIM("Label") = '' THEN NULL
            WHEN "Label" ILIKE '%BENIGN%' THEN 0
            ELSE 1
        END AS is_attack
    FROM stg_cicids_raw
""")

print(f"✅ stg_cicids created with normalized labels")

# Verify the cleanup
print("\n=== Cleaned Label Distribution ===")
print(con.execute("""
    SELECT label_clean, COUNT(*) AS cnt
    FROM stg_cicids
    GROUP BY label_clean
    ORDER BY cnt DESC
""").fetchdf())

print("\n=== Attack Family Distribution ===")
print(con.execute("""
    SELECT attack_family, 
           COUNT(*) AS cnt,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM stg_cicids), 2) AS pct
    FROM stg_cicids
    GROUP BY attack_family
    ORDER BY cnt DESC
""").fetchdf())

print("\n=== Binary Class Balance (excluding Unlabeled) ===")
print(con.execute("""
    SELECT 
        CASE WHEN is_attack = 0 THEN 'Benign' 
             WHEN is_attack = 1 THEN 'Attack' 
             ELSE 'Unlabeled' END AS class,
        COUNT(*) AS cnt
    FROM stg_cicids
    GROUP BY is_attack
    ORDER BY cnt DESC
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ stg_cicids created with normalized labels

=== Cleaned Label Distribution ===
                   label_clean      cnt
0                       BENIGN  2273097
1                    Unlabeled   288602
2                     DoS Hulk   231073
3                     PortScan   158930
4                         DDoS   128027
5                DoS GoldenEye    10293
6                  FTP-Patator     7938
7                  SSH-Patator     5897
8                DoS slowloris     5796
9             DoS Slowhttptest     5499
10                         Bot     1966
11    Web Attack - Brute Force     1507
12            Web Attack - XSS      652
13                Infiltration       36
14  Web Attack - Sql Injection       21
15                  Heartbleed       11

=== Attack Family Distribution ===
    attack_family      cnt    pct
0          Benign  2273097  72.87
1       Unlabeled   288602   9.25
2             DoS   252661   8.10
3  Reconnaissance   158930   5.09
4            DDoS   128027   4.10


# Resolving type issues

In [42]:
import duckdb
from pathlib import Path

DUCKDB_PATH = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\interim\cicids_eda.duckdb")
con = duckdb.connect(str(DUCKDB_PATH))

# All columns are currently VARCHAR — let's confirm
schema = con.execute("DESCRIBE stg_cicids").fetchdf()
print(f"Total columns: {len(schema)}")
print(f"\nColumn types currently:")
print(schema['column_type'].value_counts())

print(f"\n=== First 15 columns ===")
print(schema[['column_name', 'column_type']].head(15))

Total columns: 88

Column types currently:
column_type
VARCHAR    87
INTEGER     1
Name: count, dtype: int64

=== First 15 columns ===
                    column_name column_type
0                       Flow ID     VARCHAR
1                     Source IP     VARCHAR
2                   Source Port     VARCHAR
3                Destination IP     VARCHAR
4              Destination Port     VARCHAR
5                      Protocol     VARCHAR
6                     Timestamp     VARCHAR
7                 Flow Duration     VARCHAR
8             Total Fwd Packets     VARCHAR
9        Total Backward Packets     VARCHAR
10  Total Length of Fwd Packets     VARCHAR
11  Total Length of Bwd Packets     VARCHAR
12        Fwd Packet Length Max     VARCHAR
13        Fwd Packet Length Min     VARCHAR
14       Fwd Packet Length Mean     VARCHAR


In [43]:
# These are the well-known problematic columns in CICIDS2017
problem_cols = ['Flow Bytes/s', 'Flow Packets/s']

for col in problem_cols:
    print(f"\n=== {col} — distinct values containing non-numeric content ===")
    print(con.execute(f"""
        SELECT "{col}", COUNT(*) AS cnt
        FROM stg_cicids
        WHERE "{col}" IS NULL 
           OR "{col}" ILIKE '%inf%' 
           OR "{col}" ILIKE '%nan%'
           OR TRY_CAST("{col}" AS DOUBLE) IS NULL
        GROUP BY "{col}"
        ORDER BY cnt DESC
        LIMIT 5
    """).fetchdf())


=== Flow Bytes/s — distinct values containing non-numeric content ===
  Flow Bytes/s     cnt
0         None  288622
1     Infinity    1394
2          NaN    1338
3          inf     115

=== Flow Packets/s — distinct values containing non-numeric content ===
  Flow Packets/s     cnt
0           None  288602
1       Infinity    2732
2            inf     135


In [44]:
# Define column groups with their target types
column_types = {
    # Network identifiers (keep as VARCHAR/INT)
    'Flow ID': 'VARCHAR',
    'Source IP': 'VARCHAR',
    'Destination IP': 'VARCHAR',
    'Source Port': 'INTEGER',
    'Destination Port': 'INTEGER',
    'Protocol': 'INTEGER',
    
    # Time
    # 'Timestamp' handled separately in Cell 6.4
    
    # Flow metrics — INTEGER
    'Flow Duration': 'INTEGER',
    'Total Fwd Packets': 'INTEGER',
    'Total Backward Packets': 'INTEGER',
    'Total Length of Fwd Packets': 'INTEGER',
    'Total Length of Bwd Packets': 'INTEGER',
    'Fwd Packet Length Max': 'INTEGER',
    'Fwd Packet Length Min': 'INTEGER',
    'Bwd Packet Length Max': 'INTEGER',
    'Bwd Packet Length Min': 'INTEGER',
    'Flow IAT Max': 'BIGINT',
    'Flow IAT Min': 'BIGINT',
    'Fwd IAT Total': 'BIGINT',
    'Fwd IAT Max': 'BIGINT',
    'Fwd IAT Min': 'BIGINT',
    'Bwd IAT Total': 'BIGINT',
    'Bwd IAT Max': 'BIGINT',
    'Bwd IAT Min': 'BIGINT',
    'Fwd PSH Flags': 'INTEGER',
    'Bwd PSH Flags': 'INTEGER',
    'Fwd URG Flags': 'INTEGER',
    'Bwd URG Flags': 'INTEGER',
    'Fwd Header Length': 'INTEGER',
    'Bwd Header Length': 'INTEGER',
    'Min Packet Length': 'INTEGER',
    'Max Packet Length': 'INTEGER',
    'FIN Flag Count': 'INTEGER',
    'SYN Flag Count': 'INTEGER',
    'RST Flag Count': 'INTEGER',
    'PSH Flag Count': 'INTEGER',
    'ACK Flag Count': 'INTEGER',
    'URG Flag Count': 'INTEGER',
    'CWE Flag Count': 'INTEGER',
    'ECE Flag Count': 'INTEGER',
    'Down/Up Ratio': 'INTEGER',
    'Fwd Avg Bytes/Bulk': 'INTEGER',
    'Fwd Avg Packets/Bulk': 'INTEGER',
    'Fwd Avg Bulk Rate': 'INTEGER',
    'Bwd Avg Bytes/Bulk': 'INTEGER',
    'Bwd Avg Packets/Bulk': 'INTEGER',
    'Bwd Avg Bulk Rate': 'INTEGER',
    'Subflow Fwd Packets': 'INTEGER',
    'Subflow Fwd Bytes': 'INTEGER',
    'Subflow Bwd Packets': 'INTEGER',
    'Subflow Bwd Bytes': 'INTEGER',
    'Init_Win_bytes_forward': 'INTEGER',
    'Init_Win_bytes_backward': 'INTEGER',
    'act_data_pkt_fwd': 'INTEGER',
    'min_seg_size_forward': 'INTEGER',
    'Active Max': 'BIGINT',
    'Active Min': 'BIGINT',
    'Idle Max': 'BIGINT',
    'Idle Min': 'BIGINT',
    
    # Floating point — DOUBLE (handles Inf as NULL via NULLIF)
    'Fwd Packet Length Mean': 'DOUBLE',
    'Fwd Packet Length Std': 'DOUBLE',
    'Bwd Packet Length Mean': 'DOUBLE',
    'Bwd Packet Length Std': 'DOUBLE',
    'Flow Bytes/s': 'DOUBLE',
    'Flow Packets/s': 'DOUBLE',
    'Flow IAT Mean': 'DOUBLE',
    'Flow IAT Std': 'DOUBLE',
    'Fwd IAT Mean': 'DOUBLE',
    'Fwd IAT Std': 'DOUBLE',
    'Bwd IAT Mean': 'DOUBLE',
    'Bwd IAT Std': 'DOUBLE',
    'Fwd Packets/s': 'DOUBLE',
    'Bwd Packets/s': 'DOUBLE',
    'Packet Length Mean': 'DOUBLE',
    'Packet Length Std': 'DOUBLE',
    'Packet Length Variance': 'DOUBLE',
    'Average Packet Size': 'DOUBLE',
    'Avg Fwd Segment Size': 'DOUBLE',
    'Avg Bwd Segment Size': 'DOUBLE',
    'Active Mean': 'DOUBLE',
    'Active Std': 'DOUBLE',
    'Idle Mean': 'DOUBLE',
    'Idle Std': 'DOUBLE',
}

# Build the SELECT clause with TRY_CAST for safe conversion
# For DOUBLE columns, also handle 'Infinity'/'-Infinity'/'NaN' strings
def cast_clause(col, target_type):
    if target_type == 'VARCHAR':
        return f'"{col}"'
    elif target_type == 'DOUBLE':
        # Handle Infinity/NaN strings explicitly
        return f"""CASE 
            WHEN "{col}" ILIKE '%inf%' OR "{col}" ILIKE '%nan%' THEN NULL
            ELSE TRY_CAST("{col}" AS DOUBLE)
        END AS "{col}" """
    else:
        return f'TRY_CAST("{col}" AS {target_type}) AS "{col}"'

cast_clauses = [cast_clause(c, t) for c, t in column_types.items()]

# Add timestamp (parse it) and the metadata columns we keep as VARCHAR
cast_clauses.append('"Timestamp" AS event_time_raw')  # we'll parse in next cell
cast_clauses.append('label_clean')
cast_clauses.append('attack_family')
cast_clauses.append('TRY_CAST(is_attack AS INTEGER) AS is_attack')
cast_clauses.append('source_day')

select_sql = ',\n    '.join(cast_clauses)

con.execute(f"""
    CREATE OR REPLACE TABLE stg_cicids_typed AS
    SELECT 
        {select_sql}
    FROM stg_cicids
""")

# Verify
new_schema = con.execute("DESCRIBE stg_cicids_typed").fetchdf()
print(f"✅ stg_cicids_typed created: {len(new_schema)} columns")
print(f"\nType distribution:")
print(new_schema['column_type'].value_counts())

cnt = con.execute("SELECT COUNT(*) FROM stg_cicids_typed").fetchone()[0]
print(f"\nRow count: {cnt:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ stg_cicids_typed created: 87 columns

Type distribution:
column_type
INTEGER    44
DOUBLE     24
BIGINT     12
VARCHAR     7
Name: count, dtype: int64

Row count: 3,119,345


# Parsing Timestamps

In [45]:
# First, peek at distinct timestamp formats in the data
print("=== Sample timestamps from each day ===")
print(con.execute("""
    SELECT source_day, event_time_raw, COUNT(*) AS cnt
    FROM stg_cicids_typed
    GROUP BY source_day, event_time_raw
    ORDER BY source_day, cnt DESC
    LIMIT 20
""").fetchdf())

=== Sample timestamps from each day ===
   source_day event_time_raw    cnt
0      Friday  7/7/2017 2:55  46216
1      Friday  7/7/2017 2:52  44127
2      Friday  7/7/2017 2:54  36020
3      Friday  7/7/2017 2:51  11205
4      Friday  7/7/2017 4:13  11188
5      Friday  7/7/2017 3:57  10769
6      Friday  7/7/2017 3:58   9731
7      Friday  7/7/2017 3:59   9460
8      Friday  7/7/2017 4:11   9438
9      Friday  7/7/2017 4:01   9281
10     Friday  7/7/2017 4:14   9254
11     Friday  7/7/2017 4:09   9173
12     Friday  7/7/2017 4:00   9164
13     Friday  7/7/2017 4:15   9042
14     Friday  7/7/2017 4:08   8930
15     Friday  7/7/2017 4:12   8847
16     Friday  7/7/2017 4:07   8799
17     Friday  7/7/2017 4:03   8788
18     Friday  7/7/2017 4:05   8556
19     Friday  7/7/2017 4:10   8538


In [46]:
print("=== Date prefix distribution by source_day ===")
print(con.execute("""
    SELECT 
        source_day,
        SUBSTR(event_time_raw, 1, 8) AS date_part,
        COUNT(*) AS cnt
    FROM stg_cicids_typed
    WHERE event_time_raw IS NOT NULL
    GROUP BY source_day, date_part
    ORDER BY source_day, cnt DESC
""").fetchdf().head(30))

=== Date prefix distribution by source_day ===
  source_day date_part     cnt
0     Friday  7/7/2017  703245
1     Monday  03/07/20  529918
2   Thursday  6/7/2017  458968
3    Tuesday  4/7/2017  445909
4  Wednesday  5/7/2017  692703


In [47]:
print("=== Hour distribution (extracted from string) ===")
print(con.execute("""
    SELECT 
        source_day,
        TRY_CAST(SPLIT_PART(SPLIT_PART(event_time_raw, ' ', 2), ':', 1) AS INTEGER) AS hour_value,
        COUNT(*) AS cnt
    FROM stg_cicids_typed
    WHERE event_time_raw IS NOT NULL
    GROUP BY source_day, hour_value
    ORDER BY source_day, hour_value
""").fetchdf())

=== Hour distribution (extracted from string) ===
   source_day  hour_value     cnt
0      Friday           1   43037
1      Friday           2  191591
2      Friday           3  104339
3      Friday           4  171693
4      Friday           5    1552
5      Friday           8       2
6      Friday           9   49983
7      Friday          10   71574
8      Friday          11   55425
9      Friday          12   14049
10     Monday           1   57600
11     Monday           2   48567
12     Monday           3   48916
13     Monday           4   65875
14     Monday           5    1841
15     Monday           8     809
16     Monday           9   74418
17     Monday          10   85079
18     Monday          11   88872
19     Monday          12   57941
20   Thursday           1   50104
21   Thursday           2   57695
22   Thursday           3  129511
23   Thursday           4   48825
24   Thursday           5    2467
25   Thursday           8       6
26   Thursday           9   6231

In [51]:
import duckdb
from pathlib import Path

DUCKDB_PATH = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\interim\cicids_eda.duckdb")
con = duckdb.connect(str(DUCKDB_PATH))

# Get a few sample full timestamps from each day
print("=== Full timestamp samples per day ===\n")
for day in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']:
    samples = con.execute(f"""
        SELECT event_time_raw, COUNT(*) AS cnt
        FROM stg_cicids_typed
        WHERE source_day = '{day}'
          AND event_time_raw IS NOT NULL
          AND TRIM(event_time_raw) != ''
        GROUP BY event_time_raw
        ORDER BY RANDOM()
        LIMIT 5
    """).fetchdf()
    print(f"--- {day} ---")
    for _, row in samples.iterrows():
        print(f"  {repr(row['event_time_raw'])} (cnt: {row['cnt']:,})")
    print()

=== Full timestamp samples per day ===

--- Monday ---
  '03/07/2017 11:47:00' (cnt: 67)
  '03/07/2017 10:48:05' (cnt: 4)
  '03/07/2017 11:28:21' (cnt: 5)
  '03/07/2017 03:45:44' (cnt: 3)
  '03/07/2017 03:42:06' (cnt: 1)

--- Tuesday ---
  '4/7/2017 1:52' (cnt: 739)
  '4/7/2017 12:22' (cnt: 156)
  '4/7/2017 10:30' (cnt: 527)
  '4/7/2017 11:43' (cnt: 627)
  '4/7/2017 11:33' (cnt: 519)

--- Wednesday ---
  '5/7/2017 10:59' (cnt: 12,934)
  '5/7/2017 3:37' (cnt: 123)
  '5/7/2017 9:23' (cnt: 2,729)
  '5/7/2017 3:22' (cnt: 1,630)
  '5/7/2017 11:22' (cnt: 965)

--- Thursday ---
  '6/7/2017 10:35' (cnt: 1,092)
  '6/7/2017 11:04' (cnt: 197)
  '6/7/2017 12:19' (cnt: 766)
  '6/7/2017 4:50' (cnt: 1,471)
  '6/7/2017 4:58' (cnt: 1,972)

--- Friday ---
  '7/7/2017 11:21' (cnt: 659)
  '7/7/2017 10:49' (cnt: 203)
  '7/7/2017 4:42' (cnt: 816)
  '7/7/2017 9:35' (cnt: 349)
  '7/7/2017 9:37' (cnt: 1,305)



In [52]:
# Decompose the parser to find which step fails
print("=== Step-by-step parsing test on a sample row ===")
print(con.execute("""
    SELECT 
        event_time_raw AS original,
        SPLIT_PART(event_time_raw, '/', 1) AS day_part,
        SPLIT_PART(event_time_raw, '/', 2) AS month_part,
        SPLIT_PART(event_time_raw, '/', 3) AS year_and_time_part,
        SPLIT_PART(event_time_raw, ' ', 2) AS time_part,
        SPLIT_PART(SPLIT_PART(event_time_raw, ' ', 2), ':', 1) AS hour_part,
        SPLIT_PART(SPLIT_PART(event_time_raw, ' ', 2), ':', 2) AS minute_part
    FROM stg_cicids_typed
    WHERE event_time_raw IS NOT NULL AND TRIM(event_time_raw) != ''
    LIMIT 5
""").fetchdf().to_string())

=== Step-by-step parsing test on a sample row ===
        original day_part month_part year_and_time_part time_part hour_part minute_part
0  7/7/2017 3:30        7          7          2017 3:30      3:30         3          30
1  7/7/2017 3:30        7          7          2017 3:30      3:30         3          30
2  7/7/2017 3:30        7          7          2017 3:30      3:30         3          30
3  7/7/2017 3:30        7          7          2017 3:30      3:30         3          30
4  7/7/2017 3:30        7          7          2017 3:30      3:30         3          30


In [54]:
import duckdb
from pathlib import Path

DUCKDB_PATH = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\interim\cicids_eda.duckdb")
con = duckdb.connect(str(DUCKDB_PATH))

con.execute("""
    CREATE OR REPLACE TABLE stg_cicids_final AS
    SELECT *,
        CASE
            -- NULL guard
            WHEN event_time_raw IS NULL OR TRIM(event_time_raw) = '' THEN NULL
            
            -- Monday: '03/07/2017 11:47:00' (zero-padded, with seconds, 12-hour)
            WHEN source_day = 'Monday' THEN
                CASE
                    WHEN TRY_CAST(SPLIT_PART(SPLIT_PART(TRIM(event_time_raw), ' ', 2), ':', 1) AS INTEGER) BETWEEN 1 AND 5
                    THEN TRY_STRPTIME(TRIM(event_time_raw), '%d/%m/%Y %H:%M:%S') + INTERVAL 12 HOUR
                    ELSE TRY_STRPTIME(TRIM(event_time_raw), '%d/%m/%Y %H:%M:%S')
                END
            
            -- Tue/Wed/Thu/Fri: '7/7/2017 3:30' (no padding, no seconds, 12-hour)
            ELSE
                CASE
                    WHEN TRY_CAST(SPLIT_PART(SPLIT_PART(TRIM(event_time_raw), ' ', 2), ':', 1) AS INTEGER) BETWEEN 1 AND 5
                    THEN TRY_STRPTIME(TRIM(event_time_raw), '%-d/%-m/%Y %-H:%M') + INTERVAL 12 HOUR
                    ELSE TRY_STRPTIME(TRIM(event_time_raw), '%-d/%-m/%Y %-H:%M')
                END
        END AS event_time
    FROM stg_cicids_typed
""")

print("✅ stg_cicids_final created (timestamp parser corrected for Monday)")

# Verify
print("\n=== Parse summary by day ===")
print(con.execute("""
    SELECT 
        source_day,
        MIN(event_time) AS earliest,
        MAX(event_time) AS latest,
        EXTRACT(HOUR FROM MIN(event_time)) AS earliest_hour,
        EXTRACT(HOUR FROM MAX(event_time)) AS latest_hour,
        COUNT(event_time) AS parsed_count,
        SUM(CASE WHEN event_time_raw IS NULL OR TRIM(event_time_raw) = '' THEN 1 ELSE 0 END) AS null_input,
        SUM(CASE 
                WHEN event_time IS NULL 
                AND event_time_raw IS NOT NULL 
                AND TRIM(event_time_raw) != ''
                THEN 1 ELSE 0 
            END) AS parse_failed
    FROM stg_cicids_final
    GROUP BY source_day
    ORDER BY MIN(event_time)
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ stg_cicids_final created (timestamp parser corrected for Monday)

=== Parse summary by day ===
  source_day            earliest              latest  earliest_hour  latest_hour  parsed_count  null_input  parse_failed
0     Monday 2017-07-03 08:55:58 2017-07-03 17:01:34              8           17        529918         0.0           0.0
1    Tuesday 2017-07-04 08:53:00 2017-07-04 17:00:00              8           17        445909         0.0           0.0
2  Wednesday 2017-07-05 08:42:00 2017-07-05 17:10:00              8           17        692703         0.0           0.0
3   Thursday 2017-07-06 08:59:00 2017-07-06 17:04:00              8           17        458968    288602.0           0.0
4     Friday 2017-07-07 08:59:00 2017-07-07 17:02:00              8           17        703245         0.0           0.0


In [55]:
print("=== Attack timing validation against documentation ===\n")
print(con.execute("""
    SELECT 
        source_day,
        label_clean,
        MIN(event_time) AS first_seen,
        MAX(event_time) AS last_seen,
        COUNT(*) AS events
    FROM stg_cicids_final
    WHERE attack_family NOT IN ('Benign', 'Unlabeled')
    GROUP BY source_day, label_clean
    ORDER BY source_day, MIN(event_time)
""").fetchdf().to_string())

=== Attack timing validation against documentation ===

   source_day                 label_clean          first_seen           last_seen  events
0      Friday                         Bot 2017-07-07 09:34:00 2017-07-07 12:59:00    1966
1      Friday                    PortScan 2017-07-07 13:05:00 2017-07-07 15:23:00  158930
2      Friday                        DDoS 2017-07-07 15:56:00 2017-07-07 16:16:00  128027
3    Thursday    Web Attack - Brute Force 2017-07-06 09:15:00 2017-07-06 10:00:00    1507
4    Thursday            Web Attack - XSS 2017-07-06 10:15:00 2017-07-06 10:35:00     652
5    Thursday  Web Attack - Sql Injection 2017-07-06 10:40:00 2017-07-06 10:42:00      21
6    Thursday                Infiltration 2017-07-06 14:19:00 2017-07-06 15:45:00      36
7     Tuesday                 FTP-Patator 2017-07-04 09:17:00 2017-07-04 10:30:00    7938
8     Tuesday                 SSH-Patator 2017-07-04 14:09:00 2017-07-04 15:11:00    5897
9   Wednesday               DoS slowloris 20